In [2]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to DuckDB
con = duckdb.connect("../data/olist.duckdb", read_only=True)     # read-only to prevent accidental writes

# Helper to run a query and get a DataFrame
def q(sql):
    return con.execute(sql).fetchdf()

# 1. Analysis
This notebook will be answering the five business questions defined in `docs/data_model.md`, using the star schema built in `02_cleaning_model.ipynb`. Findings from this notebook will feed into a Power BI dashboard and the stakeholder report.

**Main question**: *How does delivery performance vary across regions, sellers, and product categories, and how does it impact customer satisfaction?*

## Q1. Olist's order fulfiment rate
**Question**: *"What is Olist's order fulfilment rate?"*

**Why it matters**: Knowing Olist's order fulfilment rate shows a company's overall performance and will help identify problem areas where improvements can be made.

**Method**: To answer this question, we will only be using `fact_deliveries` by calculating the percentage of on-time deliveries to the total number of deliveries made. These are the conditions that qualifies a delivery as "fulfiled":
- `order_status` is `delivered`
- `estimated_delivery_date_key` is not empty
- `is_late` is False

### Queries

In [3]:
# Get count of total deliveries
total = con.execute("""
    SELECT COUNT(*) AS total_deliveries
    FROM mart.fact_deliveries
    WHERE is_delivery_complete = TRUE
""").fetchone()[0]                                              

# Get count of on-time deliveries
on_time = con.execute("""
    SELECT COUNT (*) AS on_time
    FROM mart.fact_deliveries
    WHERE is_delivery_complete = TRUE
                      AND NOT is_late     
""").fetchone()[0]

# Calculate order fulfilment rate
fulfilment_rate = on_time / total * 100

print(f"On-time deliveries:  {on_time:,}")
print(f"Total completed:     {total:,}")
print(f"Fulfilment rate:     {fulfilment_rate:.2f}%")

On-time deliveries:  101,460
Total completed:     110,173
Fulfilment rate:     92.09%


Based on this result we can see that Olist had a total of 110,173 successful deliveries and 101,460 on-time deliveries. This gives us a 92.09% of order fulfilment rate. This means that there were 8,713 deliveries that were delivered late. We dig deeper into this number below:

In [4]:
# Check late deliveries
late_breakdown = con.execute("""
    SELECT 
                             dd.year AS delivery_year,
                             COUNT (*) AS late_deliveries
    FROM mart.fact_deliveries f
    JOIN mart.dim_date dd ON f.delivery_date_key = dd.date_key
    WHERE is_late = TRUE
    GROUP BY dd.year
    ORDER BY dd.year
""").fetchdf()

late_breakdown

,delivery_year,late_deliveries
0,2016,6
1,2017,2419
2,2018,6290


Based on this result, we can see that there 6 late deliveries in 2016. In 2017 there were 2,419 late deliveries, and 6,290 in 2018. Each year the number goes up significantly than the last. A 2,400+ increase of late deliveries from 2016 to 2017, and almost 4,000 increment of late deliveries from 2017 to 2018. 

However, we also need to see the **late rate per year** instead of just the raw count.

In [5]:
late_rate_by_year = con.execute("""
    SELECT 
        dd.year,
        COUNT(*) AS total_completed,
        COUNT(*) FILTER (WHERE is_late) AS late_deliveries,
        100.0 * COUNT(*) FILTER (WHERE is_late) / COUNT(*) AS late_rate_pct
    FROM mart.fact_deliveries f
    JOIN mart.dim_date dd ON f.delivery_date_key = dd.date_key
    WHERE is_delivery_complete = TRUE
    GROUP BY dd.year
    ORDER BY dd.year
""").fetchdf()

late_rate_by_year

,year,total_completed,late_deliveries,late_rate_pct
0,2016,317,6,1.892744
1,2017,46787,2418,5.168102
2,2018,63069,6289,9.971618


Based on this result we can see that Olist's logistics performance really was degrading. The number of deliveries in 2016 was significantly lower than 2017 and 2018, due to the data starting in September. The real performance to look at is between 2017 and 2018 where the rate of late deliveries almost doubled from 5% to 9.9%. This considerable increase in rate is worth further investigation to check whether there was a problem or not.

### Analysis
Overall, Olist has a good order fulfilment rate of 92.09%. Out of 110,173 orders sucessfuly delivered, 101,460 were delivered on time. That is 8,713 of the deliveries were late. 92% of order fulfilment rate is not a bad number. However Olist should aim to be as close to 100% as possible. 

Digging deeper into these deliveries we can see that there were 6 late deliveries in 2016, 2,419 in 2017, and 6,290 in 2018. The late deliveries increased each year significantly by the thousands. However, 2016's data incomplete, so it's not a reliable source of information.

Looking further into the rate of late deliveries we can see that between 2017 and 2018, the rate of late deliveries almost doubled from 5% to 9.9%. This considerable increase is worth looking deeper into to figure out the cause.

## Q2. Olist's Delivery Reliability by State
**Question**: *"Which state has the most and least reliable delivery?"*

**Why it matters**: Knowing which state has the best delivery performance could help provide insight on what works which could be applied to other locations. Knowing which state has the worst delivery performance could help identify areas where further investigations and improvements should be implemented.

**Method**: To answer this question, we will be using `fact_deliveries` to get the delivery information joined to `dim_customer` to get the delivery locations. "Reliable delivery" means will be dedfined as on-time delivery (delivered on or berfore the estimated date), which is basically `is_late = FALSE`. The data will be filtered to `is_delivery_complete = TRUE` to focus on completed records.

### Queries

In [6]:
# 
overall_state_reliability = con.execute("""
    SELECT 
                                c.customer_state,
                                COUNT(*) as total_delivered,
                                COUNT(*) FILTER (WHERE NOT is_late) as on_time_delivery,
                                ROUND(100.0 * COUNT(*) FILTER (WHERE NOT is_late) / COUNT (*),2) as reliability_pct
    FROM mart.fact_deliveries f
    JOIN mart.dim_customers c ON f.customer_key = c.customer_key
    WHERE is_delivery_complete
    GROUP BY c.customer_state
    ORDER BY reliability_pct DESC                 
""").fetchdf()

overall_state_reliability

,customer_state,total_delivered,on_time_delivery,reliability_pct
0,AC,91,88,96.70
1,RO,273,262,95.97
2,AM,163,156,95.71
3,PR,5649,5379,95.22
4,AP,81,77,95.06
5,MG,12913,12210,94.56
6,SP,46435,43757,94.23
7,MT,1037,967,93.25
8,RS,6131,5709,93.12
9,DF,2355,2180,92.57


Based on this result we can see that there are 26 states on record, with the **highest percentage being 96.7% for `AC` state**, with 88 out of 91 deliveries to that state being on time. The **lowest reliability seems to be from the state `AL` at 75.8%**, with 324 out of 427 deliveries being on time. 

However, it is important to also take the number of deliveries into account. While `AC` has the highest reliability percentage, its total deliveries did not even surpass 100 where most of the other states did. To make it more insightful, we can filter the data to only show states with deliveries above 1,000 and then sort their reliability again..

In [7]:
# Resorting by number of deliveries
state_reliability_sorted_deliveries = con.execute("""
    SELECT 
                                c.customer_state,
                                COUNT(*) as total_delivered,
                                COUNT(*) FILTER (WHERE NOT is_late) as on_time_delivery,
                                ROUND(100.0 * COUNT(*) FILTER (WHERE NOT is_late) / COUNT (*),2) as reliability_pct
    FROM mart.fact_deliveries f
    JOIN mart.dim_customers c ON f.customer_key = c.customer_key
    WHERE is_delivery_complete 
    GROUP BY c.customer_state          
    HAVING total_delivered >= 1000
    ORDER BY reliability_pct DESC
""").fetchdf()

state_reliability_sorted_deliveries

,customer_state,total_delivered,on_time_delivery,reliability_pct
0,PR,5649,5379,95.22
1,MG,12913,12210,94.56
2,SP,46435,43757,94.23
3,MT,1037,967,93.25
4,RS,6131,5709,93.12
5,DF,2355,2180,92.57
6,GO,2277,2098,92.14
7,SC,4097,3703,90.38
8,PE,1746,1568,89.81
9,ES,2225,1953,87.78


### Analysis

Applying a 1,000-delivery threshold reduces the result to 13 states with more meaningful volumes. Reliability ranges from 84.70% (CE) to 95.22% (PR)

PR, MG, and SP lead with 94%+ reliability, all above 5,000 deliveries, which quite substantial. Especially SP, with 46,435 deliveries, is by far the largest market and maintained a 94.23% on-time performance.

At the bottom, CE, BA, PA, and ES form the low-reliability states. This result is consistent with the region's weaker logistics infrastructure.

State RJ, however, is the anomaly. With 14,140 deliveries, its reliability is only at 87.02%. It underperformed significantly compared to the other high-volume markets (SP at 94.23% and MG at 94.56%). Because of its size, RJ has a large impact on Olist's overall on-time rate. Roughly 1,800 late deliveries from this siingle state alone. This warants further investigation into causes specific to RJ.



## Q3. Olist's Delivery Reliability by Product Category
**Question**: *"Which product category has the highest and lowest rate of on-time deliveries?"*

**Why it matters**: Knowing which category of product has the best and worst delivery performance could provide insight on what impact does a product type and size have on on-time delivery.

**Method**: To answer this question, we will be using `fact_deliveries` to get the delivery information joined to `dim_products` to get the product category names and dimensions. "Reliable delivery" means will be defined as on-time delivery (delivered on or berfore the estimated date), which is basically `is_late = FALSE`. The data will be filtered to `is_delivery_complete = TRUE` to focus on completed records.

### Queries
#### Reliability by Category Name

In [8]:
overall_reliability_by_product = con.execute("""
                                     SELECT 
                                        p.product_category_name_english,
                                        COUNT(*) AS total_delivered,
                                        COUNT(*) FILTER (WHERE NOT f.is_late) AS on_time_delivery,
                                        ROUND(100.0 * COUNT(*) FILTER (WHERE NOT f.is_late) / COUNT (*),2) AS reliability_pct
                                     FROM mart.fact_deliveries f
                                     JOIN mart.dim_products p ON f.product_key = p.product_key
                                     WHERE p.has_category 
                                     GROUP BY p.product_category_name_english
                                     ORDER BY reliability_pct DESC, p.product_category_name_english ASC
""").fetchdf()

overall_reliability_by_product

,product_category_name_english,total_delivered,on_time_delivery,reliability_pct
0,cds_dvds_musicals,14,14,100.00
1,la_cuisine,14,14,100.00
2,security_and_services,2,2,100.00
3,flowers,33,32,96.97
4,costruction_tools_tools,103,97,94.17
...,...,...,...,...
67,audio,364,316,86.81
68,christmas_supplies,153,132,86.27
69,fashion_underwear_beach,131,111,84.73
70,furniture_mattress_and_upholstery,38,32,84.21


We start by looking at the reliability based on product category name alone. The reliability ranges from 83.33% (`home_comfort_2`) to a full 100% (`cds_dvds_musicals`, `la_cuisine`, and `security_services`). 

The top 5 categories all have 94%+ reliability, with the top 3 at 100%, which is incredible. However, we can see that the delivery volume is much too few. All less than 15 deliveries, and `security_and_services` only has 2 deliveries. Meanwhile `flowers`, with 33 deliveries have 96.97% reliability and `construction_tools_tools` at 94.17 reliability with 103 deliveries.

The bottom 5 categories are all below 87% reliability, with `home_comfort_2` being the lowest at 83.33% with 30 deliveries. 

With this initial glance, it seems that breaking the reliability down simply by the category name alone is not enough. As we can see, the delivery volumes of the most reliable categories are too small. We can try sorting based on the delivery volumes.

In [9]:
overall_reliability_by_product = con.execute("""
                                     SELECT 
                                        p.product_category_name_english,
                                        COUNT(*) AS total_delivered,
                                        COUNT(*) FILTER (WHERE NOT f.is_late) AS on_time_delivery,
                                        ROUND(100.0 * COUNT(*) FILTER (WHERE NOT f.is_late) / COUNT (*),2) AS reliability_pct
                                     FROM mart.fact_deliveries f
                                     JOIN mart.dim_products p ON f.product_key = p.product_key
                                     WHERE p.has_category 
                                     GROUP BY p.product_category_name_english
                                    HAVING total_delivered >= 1000
                                     ORDER BY total_delivered DESC, p.product_category_name_english ASC
""").fetchdf()

overall_reliability_by_product

,product_category_name_english,total_delivered,on_time_delivery,reliability_pct
0,bed_bath_table,11115,10033,90.27
1,health_beauty,9670,8609,89.03
2,sports_leisure,8641,7806,90.34
3,furniture_decor,8334,7472,89.66
4,computers_accessories,7827,7049,90.06
5,housewares,6964,6354,91.24
6,watches_gifts,5991,5372,89.67
7,telephony,4545,4061,89.35
8,garden_tools,4347,3928,90.36
9,auto,4235,3796,89.63


Sorting by delivery volume and filtering to only products with 1000+ deliveries now shows us a more insightful result. We now have 21 product categories ranging from 1,092 deliveries (`luggage_accessories`) to 11,115 deliveries (`bed_bath_table`). 

Reliability seems to range from 88.74% (`baby`) to 93.32% (`luggage_accessories`). A 4.6 percentage difference. This gap is much narrower compared to the state-level variation observed in Q2 (around 20 percentage points). This suggests that product category may not be a strong predictor of delivery reliability than destination geography.

The highest-volume category, `bed_bath_table` (11,115 deliveries) sits at the middle of the reliability range at 90.27%. No clear pattern of product category or implied product size to on-time performance. `furniture_decor` (89.66%) and `office_furniture` (89.83%) are not at the bottom of the rankings despite containing larger, harder-to-ship items.

A follow-up analysis using actual product dimensions rather than category names would be useful to test whether physical size predicts delivery reliability.

### Q3b. Does Product Size Predict Delivery Reliability?
**Question**: *"Which product size category has the highest and lowest rate of on-time deliveries?"*

**Why it matters**: Knowing which category of product size has the best and worst delivery performance could provide insight on whether an item's size have any meaningful impact on reliability.

**Method**: To answer this question, we will be using `fact_deliveries` to get the delivery information joined to `dim_products` to get the product dimensions. "Reliable delivery" means will be defined as on-time delivery (delivered on or berfore the estimated date), which is basically `is_late = FALSE`. The data will be filtered to `is_delivery_complete = TRUE` to focus on completed records.

To answer this question, we will need to derive categorical values for the products, whether they are:
small, medium, large, or bulky. To figure out the volumes that qualify into these categories, we can see the distribution of the volumes of the products being delivered first.

In [10]:
con.execute("""
    SELECT 
        MIN(product_length_cm * product_height_cm * product_width_cm) AS min_vol,
        MAX(product_length_cm * product_height_cm * product_width_cm) AS max_vol,
        PERCENTILE_CONT(0.25) WITHIN GROUP 
            (ORDER BY product_length_cm * product_height_cm * product_width_cm) AS p25,
        PERCENTILE_CONT(0.50) WITHIN GROUP 
            (ORDER BY product_length_cm * product_height_cm * product_width_cm) AS p50,
        PERCENTILE_CONT(0.75) WITHIN GROUP 
            (ORDER BY product_length_cm * product_height_cm * product_width_cm) AS p75
    FROM mart.dim_products
    WHERE has_dimensions = TRUE
""").fetchdf()

,min_vol,max_vol,p25,p50,p75
0,168,296208,2880.0,6840.0,18480.0


Based on this, we can see that the range of volumes of the products delivered by Olist. So the categorization is:
- `small`: < 3,000 cm3 (phone, book, cosmetics)
- `medium`: 3,000 - 18,000 cm3 (shoebox to small appliance range)
- `large`: 18,000 - 100,000 cm3 (microwave to large appliance)
- `bulky`: > 100,000 cm3 (large furniture, sports equipment)

In [15]:
con.execute("""
    SELECT
        CASE
            WHEN product_length_cm * product_height_cm * product_width_cm < 3000 THEN 'Small'
            WHEN product_length_cm * product_height_cm * product_width_cm < 18000 THEN 'Medium'
            WHEN product_length_cm * product_height_cm * product_width_cm < 100000 THEN 'Large'
            ELSE 'Bulky'
        END AS size_category,
        COUNT(*) AS product_count
    FROM mart.dim_products
    WHERE has_dimensions = TRUE
    GROUP BY size_category
    ORDER BY product_count DESC
""").fetchdf()

,size_category,product_count
0,Medium,15876
1,Small,8493
2,Large,7858
3,Bulky,722


Based on this result, we can see that Olist's catalog mostly contains medium-sized products (15,876 products), followed by small (8,493 products) and large (7,858 products), with bulky items being the least common (722 products). 

With this information we can now examine the delivery volumes by size category.

In [16]:
reliability_by_size = con.execute("""
                                  SELECT
                                    CASE
                                        WHEN product_length_cm * product_height_cm * product_width_cm < 3000 THEN 'Small'
                                        WHEN product_length_cm * product_height_cm * product_width_cm < 18000 THEN 'Medium'
                                        WHEN product_length_cm * product_height_cm * product_width_cm < 100000 THEN 'Large'
                                        ELSE 'Bulky'
                                    END AS size_category,
                                    COUNT(*) AS total_delivered,
                                    COUNT(*) FILTER (WHERE NOT is_late) AS on_time_delivery,
                                    ROUND(100.0 * COUNT(*) FILTER (WHERE NOT f.is_late) / COUNT(*), 2) AS reliability_pct
                                  FROM mart.fact_deliveries f
                                  JOIN mart.dim_products p ON f.product_key = p.product_key
                                  WHERE f.is_delivery_complete = TRUE AND has_dimensions = TRUE
                                  GROUP BY size_category
                                  ORDER BY reliability_pct DESC
""").fetchdf()

reliability_by_size

,size_category,total_delivered,on_time_delivery,reliability_pct
0,Medium,52257,48337,92.50
1,Small,29609,27249,92.03
2,Large,26744,24471,91.50
3,Bulky,1545,1385,89.64


**Note**: products that don't have dimension information are excluded in this analysis.

Based on this result, there is a weak but consistent pattern where reliability decreases as product size increases, from 92.50% for medium-sized products down to 89.64% for bulky ones. This is a 2.86 precentage gap. Small products (92.03%) and large products (91.50%) fall in between.

### Analysis
The first analysis, reliability by product category name, didn't show any clear pattern on the kind of products that have high delivery reliability. Further sorting by delivery volumes showed a more insightful result, where reliability across product categories are surprisingly uniform. A 4.6% difference across 21 categories suggests that what a product is doesn't really impact its delivery reliability.

A shift in analysis to reliability by product size category showed considerably better results than pure category names. With the size categorization that was made above, a weak but consistent pattern emerged, where the bigger a product is, the less reliable its delivery is going to be. A 2.86% difference spread among 4 categories means that a product size has some impact on delivery and provides more actionable insights.